# 20 - Evaluate ECG diagnosis generation and CXR augmentation

The primary endpoint is ECG clinical-content quality. CXR is evaluated only as a context source.
Fabrication is the fraction of generated positive ECG concepts absent from the held-out machine
diagnosis; omission is the fraction of reference concepts not generated. Study-macro and label-micro
rates, macro/micro F1, Hamming accuracy, per-label results, CIDEr, BERTScore, ROUGE,
BLEU-1/BLEU-4, and METEOR are reported. BLEU and METEOR are included for comparison with
prior IEEE ECG report-generation work; structured ECG-label outcomes remain primary.

Paired comparisons directly test the ECG-to-ECG neighbor baseline versus zero-shot, CXR-only versus
zero-shot, ECG+CXR versus ECG-only, CodeBind versus CLIP CXR retrieval, and retrieved versus random
CXR context. The ECG-only condition is ECG-ReGen-style but is not an exact reproduction. Automated
labels are not a clinical safety determination; blinded cardiologist review remains required.


In [ ]:
from pathlib import Path
import gc
import json
import os
import re

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
RAG_ROOT = ARTIFACT_ROOT / "rag_report_generation_ecg"
REPORT_ROOT = RAG_ROOT / "reports"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
config = json.loads((RAG_ROOT / "rag_config.json").read_text())
HF_CACHE = Path(config.get("hf_cache") or "/data/liangz2/ecg_analysis/hf_cache")
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["HF_XET_CACHE"] = str(HF_CACHE / "xet")
os.environ["MPLCONFIGDIR"] = str(RAG_ROOT / "cache" / "matplotlib")
os.environ["NLTK_DATA"] = str(RAG_ROOT / "cache" / "nltk_data")
for directory in [HF_CACHE / "hub", HF_CACHE / "xet", Path(os.environ["MPLCONFIGDIR"]),
                  Path(os.environ["NLTK_DATA"])]:
    directory.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None
    print("Optional package seaborn is not installed; figures will use the Matplotlib fallback.")
import nltk
import torch
from bert_score import BERTScorer
from nltk.corpus import wordnet
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import single_meteor_score
from pycocoevalcap.cider.cider import Cider
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score, hamming_loss, precision_recall_fscore_support

ALLOW_INCOMPLETE = os.environ.get("RAG_ALLOW_INCOMPLETE", "0").lower() in {"1", "true", "yes"}
BOOTSTRAPS = int(os.environ.get("RAG_EVAL_BOOTSTRAPS", "500"))
BERT_MODEL = os.environ.get("RAG_BERTSCORE_MODEL", "microsoft/deberta-xlarge-mnli")
BERT_BATCH = int(os.environ.get("RAG_BERTSCORE_BATCH_SIZE", "32"))
BERT_MAX_LENGTH = int(os.environ.get("RAG_BERTSCORE_MAX_LENGTH", "512"))
if BERT_BATCH < 1 or BERT_MAX_LENGTH < 8:
    raise ValueError("BERTScore batch size must be positive and max length must be at least 8")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# METEOR uses WordNet. Keep its small resource under the project artifact root
# rather than Biowulf's quota-limited home directory.
try:
    wordnet.ensure_loaded()
except LookupError:
    downloaded = nltk.download(
        "wordnet", download_dir=os.environ["NLTK_DATA"], quiet=True)
    if not downloaded:
        raise RuntimeError(
            "METEOR requires the NLTK WordNet resource. On a network-enabled node run: "
            f"python -m nltk.downloader -d {os.environ['NLTK_DATA']} wordnet")
    wordnet.ensure_loaded()

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

manifests = {m: read_jsonl(ARTIFACT_ROOT / "external" / m / "manifest.jsonl") for m in ["ecg", "cxr"]}
concept_rows = {m: read_jsonl(ARTIFACT_ROOT / "external" / m / "concepts.jsonl") for m in ["ecg", "cxr"]}
manifest_lookup = {m: {str(row["sample_id"]): row for row in values} for m, values in manifests.items()}
concept_lookup = {m: {str(row["sample_id"]): row for row in values} for m, values in concept_rows.items()}
contexts = read_jsonl(RAG_ROOT / "contexts" / "rag_contexts.jsonl")
context_lookup = {(row["condition"], str(row["sample_id"])): row for row in contexts}
if len(context_lookup) != len(contexts):
    raise ValueError("Duplicate active context keys")


In [ ]:
ECG_EVALUATION_RULE_VERSION = "ecg_generation_assertion_rules_v3"
ECG_RULES = {
    # Sinus rhythm alone does not imply that the complete ECG is normal.
    "normal_ecg": [
        r"\bnormal(?:\s+12[- ]lead)?\s+(?:ecg|electrocardiogram)\b",
        r"\b(?:ecg|electrocardiogram)\s+(?:is\s+)?within normal limits\b",
    ],
    "atrial_fibrillation": [r"\batrial fibrillation\b", r"\ba[- ]?fib(?:rillation)?\b"],
    "atrial_flutter": [r"\batrial flutter\b"],
    "tachycardia": [r"\btachycard(?:ia|ic)\b", r"\b(?:svt|v[- ]?tach)\b"],
    "bradycardia": [r"\bbradycard(?:ia|ic)\b"],
    "conduction_abnormality": [
        r"\bbundle branch block\b", r"\b(?:rbbb|lbbb|ivcd)\b",
        r"\b(?:av|atrioventricular|heart) block\b",
        r"\b(?:first|second|third)[ -]degree (?:av|atrioventricular) block\b",
        r"\bintraventricular conduction (?:delay|defect)\b",
    ],
    "ventricular_hypertrophy": [
        r"\b(?:left|right) ventricular hypertrophy\b",
        r"\bbiventricular hypertrophy\b", r"\bventricular hypertrophy\b",
        r"\b(?:lvh|rvh)\b",
    ],
    "ischemia": [
        r"\bischemi\w*\b", r"\bst[- ]t abnormalit\w*\b",
        r"\bst(?:[- ]segment)?\s+(?:depression|elevation)\b",
    ],
    "infarction": [
        r"\bmyocardial infarction\b", r"\binfarct(?:ion|ed)?\b",
        r"\b(?:stemi|nstemi)\b",
    ],
    "paced_rhythm": [
        r"\b(?:atrial|ventricular|dual[- ]chamber)?\s*paced rhythm\b",
        r"\bpacing spikes?\b", r"\bpacemaker rhythm\b",
    ],
    "low_voltage": [r"\blow[- ]voltage\b"],
    "qt_abnormality": [
        r"\b(?:prolonged|long|short)\s+qt(?:c)?(?: interval)?\b",
        r"\bqt(?:c)? prolongation\b",
    ],
}
CHEXPERT_MAP = {
    "atelectasis": "atelectasis", "cardiomegaly": "cardiomegaly",
    "consolidation": "consolidation", "edema": "edema",
    "enlarged cardiomediastinum": "enlarged_cardiomediastinum", "fracture": "fracture",
    "lung lesion": "lung_lesion", "lung opacity": "lung_opacity", "no finding": "no_finding",
    "pleural effusion": "pleural_effusion", "pleural other": "pleural_other",
    "pneumonia": "pneumonia", "pneumothorax": "pneumothorax",
    "support devices": "support_device", "support device": "support_device",
}
ECG_LABELS = sorted(ECG_RULES)
NEGATED_BEFORE = re.compile(
    r"(?:\bno\b|\bwithout\b|\bnegative for\b|\babsence of\b|\bfree of\b)"
    r"(?:\W+\w+){0,8}\W*$", re.I)
NEGATED_AFTER = re.compile(r"^\W*(?:is |are |was |were )?(?:not|absent)(?:\W|$)", re.I)
UNCERTAIN_BEFORE = re.compile(
    r"(?:\bpossible\b|\bpossibly\b|\bprobable\b|\blikely\b|\bmay\b|\bmight\b|"
    r"\bcannot exclude\b|\bcannot rule out\b|\bconcerning for\b|\bsuspicious for\b)"
    r"(?:\W+\w+){0,8}\W*$", re.I)
UNCERTAIN_AFTER = re.compile(r"^\W*(?:is |are )?(?:possible|probable|suspected)(?:\W|$)", re.I)

def assertion(text, match):
    before = text[max(0, match.start() - 120):match.start()]
    before = re.split(r"[.;\n]|\b(?:but|however|although|yet)\b", before, flags=re.I)[-1]
    after = text[match.end():match.end() + 50]
    if NEGATED_BEFORE.search(before) or NEGATED_AFTER.search(after):
        return "absent"
    if UNCERTAIN_BEFORE.search(before) or UNCERTAIN_AFTER.search(after):
        return "uncertain"
    return "present"

def extract_ecg_assertions(text):
    value = " ".join(str(text).split()).lower()
    result = {"present": set(), "absent": set(), "uncertain": set()}
    for concept, patterns in ECG_RULES.items():
        for pattern in patterns:
            for match in re.finditer(pattern, value):
                result[assertion(value, match)].add(concept)
    return result

def extract_ecg_present(text):
    present = set(extract_ecg_assertions(text)["present"])
    if "normal_ecg" in present and len(present) > 1:
        present.remove("normal_ecg")
    return present

ECG_RULE_TESTS = [
    ("Normal 12-lead ECG.", {"normal_ecg"}, set(), set()),
    ("Normal sinus rhythm. Inferior infarct, age undetermined.", {"infarction"}, set(), set()),
    ("No evidence of atrial fibrillation.", set(), {"atrial_fibrillation"}, set()),
    ("Possible left ventricular hypertrophy.", set(), set(), {"ventricular_hypertrophy"}),
    ("No acute ischemic changes.", set(), {"ischemia"}, set()),
    ("Ventricular paced rhythm.", {"paced_rhythm"}, set(), set()),
    ("No ST-segment elevation or depression.", set(), {"ischemia"}, set()),
]
for test_text, expected_present, expected_absent, expected_uncertain in ECG_RULE_TESTS:
    observed = extract_ecg_assertions(test_text)
    expected = {
        "present": expected_present, "absent": expected_absent,
        "uncertain": expected_uncertain,
    }
    if observed != expected:
        raise AssertionError(
            f"ECG evaluation-rule self-test failed for {test_text!r}: "
            f"expected={expected}, observed={observed}")
print("ECG generated-report extractor:", ECG_EVALUATION_RULE_VERSION,
      "self-tests:", len(ECG_RULE_TESTS))

def reference_ecg_labels(sample_id):
    values = concept_lookup["ecg"][sample_id].get("concepts", [])
    labels = {value.split(":", 1)[1] for value in values
              if value.startswith("present:") and value.split(":", 1)[1] in ECG_LABELS}
    if "normal_ecg" in labels and len(labels) > 1:
        labels.remove("normal_ecg")
    return labels

def reference_cxr_labels(sample_id):
    labels = manifest_lookup["cxr"][sample_id].get("labels", [])
    return {CHEXPERT_MAP[str(value).strip().lower()] for value in labels
            if str(value).strip().lower() in CHEXPERT_MAP}

CXR_TO_ECG_RELATIONS = {
    "cardiomegaly": {"ventricular_hypertrophy"},
    "enlarged_cardiomediastinum": {"ventricular_hypertrophy"},
    "support_device": {"paced_rhythm"},
}
ECG_TO_CXR_RELATIONS = {
    "ventricular_hypertrophy": {"cardiomegaly", "enlarged_cardiomediastinum"},
    "paced_rhythm": {"support_device"},
}

def context_implied_ecg(row):
    implied = set()
    for item in row["retrieved"]:
        sample_id = str(item["sample_id"])
        if item["modality"] == "ecg":
            implied.update(extract_ecg_present(manifest_lookup["ecg"][sample_id]["machine_diagnosis"]))
        else:
            for label in reference_cxr_labels(sample_id):
                implied.update(CXR_TO_ECG_RELATIONS.get(label, set()))
    return implied


In [ ]:
generation_rows = []
for model_key in config["models"]:
    path = RAG_ROOT / "generations" / f"{model_key}.jsonl"
    if path.is_file():
        expected_model_id = config["models"][model_key]
        generation_rows.extend(
            row for row in read_jsonl(path)
            if row.get("model_id") == expected_model_id)
active_signatures = {row["cohort_signature"] for row in contexts}
if len(active_signatures) != 1:
    raise ValueError(f"Expected one active cohort signature; found {active_signatures}")
generation_rows = [row for row in generation_rows
                   if row.get("status") == "ok"
                   and row.get("cohort_signature") in active_signatures
                   and row.get("generation_protocol_version") == config["generation_protocol_version"]]
if not generation_rows:
    raise ValueError("No successful current-protocol generations; run notebook 19")
keys = [(row["model_key"], row["condition"], str(row["sample_id"])) for row in generation_rows]
if len(keys) != len(set(keys)):
    raise ValueError("Duplicate successful generation keys")

context_keys = {(row["condition"], str(row["sample_id"])) for row in contexts}
expected_keys = {(model, condition, sample_id) for model in config["models"]
                 for condition, sample_id in context_keys}
observed_keys = set(keys)
missing, unexpected = sorted(expected_keys - observed_keys), sorted(observed_keys - expected_keys)
if (missing or unexpected) and not ALLOW_INCOMPLETE:
    raise RuntimeError(
        f"Incomplete generation matrix. Missing={missing[:20]}; unexpected={unexpected[:20]}. "
        "Finish notebook 19 or set RAG_ALLOW_INCOMPLETE=1 for a smoke analysis.")
print({"successful_generations": len(generation_rows), "missing": len(missing),
       "unexpected": len(unexpected), "allow_incomplete": ALLOW_INCOMPLETE})


In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
bleu_smoothing = SmoothingFunction().method1

def report_tokens(value):
    return re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)*", str(value).lower())

def lexical_generation_scores(reference, candidate):
    reference_tokens = report_tokens(reference)
    candidate_tokens = report_tokens(candidate)
    if not reference_tokens or not candidate_tokens:
        return {"bleu1": 0.0, "bleu4": 0.0, "meteor": 0.0}
    references = [reference_tokens]
    return {
        "bleu1": float(sentence_bleu(
            references, candidate_tokens, weights=(1.0,),
            smoothing_function=bleu_smoothing)),
        "bleu4": float(sentence_bleu(
            references, candidate_tokens, weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=bleu_smoothing)),
        "meteor": float(single_meteor_score(
            reference_tokens, candidate_tokens, wordnet=wordnet)),
    }

detailed = []
for row in generation_rows:
    sample_id = str(row["sample_id"])
    reference_text = manifest_lookup["ecg"][sample_id]["machine_diagnosis"]
    truth = reference_ecg_labels(sample_id)
    predicted_assertions = extract_ecg_assertions(row["generated_report"])
    predicted = set(predicted_assertions["present"])
    if "normal_ecg" in predicted and len(predicted) > 1:
        predicted.remove("normal_ecg")
    assertion_matches = set().union(*predicted_assertions.values())
    contradictory = (
        (predicted_assertions["present"] & predicted_assertions["absent"])
        | (predicted_assertions["present"] & predicted_assertions["uncertain"])
        | (predicted_assertions["absent"] & predicted_assertions["uncertain"])
    )
    context_row = context_lookup[(row["condition"], sample_id)]
    context_concepts = context_implied_ecg(context_row)
    y_true = [int(label in truth) for label in ECG_LABELS]
    y_pred = [int(label in predicted) for label in ECG_LABELS]
    fabricated, omitted = predicted - truth, truth - predicted
    tp, fp, fn = len(truth & predicted), len(fabricated), len(omitted)
    rouge = scorer.score(str(reference_text), row["generated_report"])
    lexical = lexical_generation_scores(reference_text, row["generated_report"])
    detailed.append({
        **row, "reference_text": reference_text,
        "ecg_evaluation_rule_version": ECG_EVALUATION_RULE_VERSION,
        "truth_labels": sorted(truth), "predicted_labels": sorted(predicted),
        "predicted_absent_labels": sorted(predicted_assertions["absent"]),
        "predicted_uncertain_labels": sorted(predicted_assertions["uncertain"]),
        "predicted_contradictory_labels": sorted(contradictory),
        "extractor_matched_any_assertion": bool(assertion_matches),
        "extractor_matched_present_label": bool(predicted),
        "context_implied_ecg_labels": sorted(context_concepts),
        "fabricated_labels": sorted(fabricated), "omitted_labels": sorted(omitted),
        "y_true": y_true, "y_pred": y_pred, "tp": tp, "fp": fp, "fn": fn,
        "fabrication_rate": fp / max(len(predicted), 1),
        "omission_rate": fn / max(len(truth), 1),
        "sample_hamming_accuracy": 1 - np.mean(np.not_equal(y_true, y_pred)),
        "context_supported_prediction_fraction": len(predicted & context_concepts) / max(len(predicted), 1),
        "context_attributable_fabrication_fraction": len(fabricated & context_concepts) / max(len(fabricated), 1),
        "rouge1_f1": rouge["rouge1"].fmeasure,
        "rouge2_f1": rouge["rouge2"].fmeasure,
        "rougeL_f1": rouge["rougeL"].fmeasure,
        **lexical,
    })

candidates = [row["generated_report"] for row in detailed]
references = [str(row["reference_text"]) for row in detailed]

def build_bert_scorer(use_fast_tokenizer):
    scorer = BERTScorer(
        model_type=BERT_MODEL, lang="en", batch_size=BERT_BATCH,
        device=DEVICE, rescale_with_baseline=False,
        use_fast_tokenizer=use_fast_tokenizer)
    # Some model tokenizers advertise a huge sentinel model_max_length.
    # New Rust tokenizers cannot convert that value to their bounded integer
    # type and raise OverflowError before scoring. The underlying DeBERTa
    # model supports at most 512 tokens, so cap both public and initialization
    # metadata before the first encode call.
    scorer._tokenizer.model_max_length = BERT_MAX_LENGTH
    if hasattr(scorer._tokenizer, "init_kwargs"):
        scorer._tokenizer.init_kwargs["model_max_length"] = BERT_MAX_LENGTH
    return scorer

bert_scorer = build_bert_scorer(use_fast_tokenizer=True)
try:
    _, _, bert_f1 = bert_scorer.score(
        candidates, references, batch_size=BERT_BATCH, verbose=True)
except OverflowError:
    print("Fast BERTScore tokenizer overflowed despite the length cap; retrying with the slow tokenizer.")
    del bert_scorer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    bert_scorer = build_bert_scorer(use_fast_tokenizer=False)
    _, _, bert_f1 = bert_scorer.score(
        candidates, references, batch_size=BERT_BATCH, verbose=True)
for row, value in zip(detailed, bert_f1.cpu().numpy()):
    row["bertscore_f1"] = float(value)
del bert_scorer, bert_f1
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Per-sample ECG metrics computed:", len(detailed))


In [ ]:
def cider_values(rows):
    references = {i: [str(row["reference_text"])] for i, row in enumerate(rows)}
    candidates = {i: [str(row["generated_report"])] for i, row in enumerate(rows)}
    corpus, per_sample = Cider().compute_score(references, candidates)
    return float(corpus), np.asarray(per_sample, dtype="float64")

summary_rows = []
frame_before_cider = pd.DataFrame(detailed)
for keys, group in frame_before_cider.groupby(["model_key", "condition"], sort=True):
    records = group.to_dict("records")
    y_true = np.asarray(group.y_true.tolist(), dtype="int8")
    y_pred = np.asarray(group.y_pred.tolist(), dtype="int8")
    cider, per_cider = cider_values(records)
    cider_map = {str(record["sample_id"]): float(value) for record, value in zip(records, per_cider)}
    for record in detailed:
        if (record["model_key"], record["condition"]) == keys:
            record["cider"] = cider_map[str(record["sample_id"])]
    summary_rows.append({
        "model_key": keys[0], "condition": keys[1], "n": len(group),
        "ecg_evaluation_rule_version": ECG_EVALUATION_RULE_VERSION,
        "retrieval_method": group.retrieval_method.iloc[0],
        "context_scope": group.context_scope.iloc[0],
        "fabrication_rate": float(group.fabrication_rate.mean()),
        "fabrication_rate_micro": float(group.fp.sum() / max(group.y_pred.map(sum).sum(), 1)),
        "omission_rate": float(group.omission_rate.mean()),
        "omission_rate_micro": float(group.fn.sum() / max(group.y_true.map(sum).sum(), 1)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "hamming_accuracy": float(1 - hamming_loss(y_true, y_pred)),
        "cider": cider, "bertscore_f1": float(group.bertscore_f1.mean()),
        "rouge1_f1": float(group.rouge1_f1.mean()), "rouge2_f1": float(group.rouge2_f1.mean()),
        "rougeL_f1": float(group.rougeL_f1.mean()),
        "bleu1": float(group.bleu1.mean()), "bleu4": float(group.bleu4.mean()),
        "meteor": float(group.meteor.mean()),
        "context_supported_prediction_fraction": float(group.context_supported_prediction_fraction.mean()),
        "context_attributable_fabrication_fraction": float(group.context_attributable_fabrication_fraction.mean()),
        "truth_positive_labels": int(y_true.sum()), "predicted_positive_labels": int(y_pred.sum()),
        "reports_with_any_assertion_fraction": float(group.extractor_matched_any_assertion.mean()),
        "reports_with_present_label_fraction": float(group.extractor_matched_present_label.mean()),
        "reports_with_contradictory_assertion_fraction": float(
            group.predicted_contradictory_labels.map(bool).mean()),
    })
summary = pd.DataFrame(summary_rows)
detailed_frame = pd.DataFrame(detailed)
detailed_frame.to_csv(REPORT_ROOT / "ecg_rag_metrics_detailed.csv", index=False)
summary.to_csv(REPORT_ROOT / "ecg_rag_metrics_summary.csv", index=False)
evaluation_protocol = {
    "primary_endpoint_family": "structured ECG clinical-content metrics",
    "reference_source": "held-out external ECG machine diagnosis",
    "bleu1": "mean smoothed sentence BLEU; NLTK SmoothingFunction.method1; weights=(1.0,)",
    "bleu4": "mean smoothed sentence BLEU; NLTK SmoothingFunction.method1; weights=(0.25,0.25,0.25,0.25)",
    "meteor": "mean NLTK single_meteor_score with WordNet synonym matching",
    "lexical_tokenization": "lowercase alphanumeric tokens retaining internal hyphen/apostrophe",
    "nltk_version": nltk.__version__,
    "ecg_evaluation_rule_version": ECG_EVALUATION_RULE_VERSION,
    "generation_protocol_version": config["generation_protocol_version"],
    "ecg_neighbor_baseline": config.get("ecg_neighbor_baseline_note"),
}
(REPORT_ROOT / "ecg_rag_evaluation_protocol.json").write_text(
    json.dumps(evaluation_protocol, indent=2))
extractor_coverage_columns = [
    "model_key", "condition", "n", "reports_with_any_assertion_fraction",
    "reports_with_present_label_fraction", "reports_with_contradictory_assertion_fraction",
    "truth_positive_labels", "predicted_positive_labels",
]
summary[extractor_coverage_columns].to_csv(
    REPORT_ROOT / "ecg_rag_extractor_coverage.csv", index=False)

per_label_rows = []
for keys, group in detailed_frame.groupby(["model_key", "condition"], sort=True):
    y_true = np.asarray(group.y_true.tolist(), dtype="int8")
    y_pred = np.asarray(group.y_pred.tolist(), dtype="int8")
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0)
    for label, p, r, score, count in zip(ECG_LABELS, precision, recall, f1, support):
        per_label_rows.append({
            "model_key": keys[0], "condition": keys[1], "label": label,
            "support": int(count), "precision": float(p), "recall": float(r), "f1": float(score)})
pd.DataFrame(per_label_rows).to_csv(REPORT_ROOT / "ecg_rag_per_label_metrics.csv", index=False)

query_ids = sorted({str(row["sample_id"]) for row in contexts})
prevalence = [{
    "label": label, "n_queries": len(query_ids),
    "positive_count": sum(label in reference_ecg_labels(sample_id) for sample_id in query_ids),
    "prevalence": sum(label in reference_ecg_labels(sample_id) for sample_id in query_ids) / max(len(query_ids), 1),
} for label in ECG_LABELS]
pd.DataFrame(prevalence).to_csv(REPORT_ROOT / "ecg_rag_cohort_label_prevalence.csv", index=False)
display(summary.sort_values(["model_key", "condition"]))


In [ ]:
def grouped_metrics(group):
    y_true = np.asarray(group.y_true.tolist(), dtype="int8")
    y_pred = np.asarray(group.y_pred.tolist(), dtype="int8")
    return {
        "fabrication_rate": float(group.fabrication_rate.mean()),
        "fabrication_rate_micro": float(group.fp.sum() / max(group.y_pred.map(sum).sum(), 1)),
        "omission_rate": float(group.omission_rate.mean()),
        "omission_rate_micro": float(group.fn.sum() / max(group.y_true.map(sum).sum(), 1)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "hamming_accuracy": float(1 - hamming_loss(y_true, y_pred)),
        "cider": float(group.cider.mean()), "bertscore_f1": float(group.bertscore_f1.mean()),
        "rouge1_f1": float(group.rouge1_f1.mean()), "rouge2_f1": float(group.rouge2_f1.mean()),
        "rougeL_f1": float(group.rougeL_f1.mean()),
        "bleu1": float(group.bleu1.mean()), "bleu4": float(group.bleu4.mean()),
        "meteor": float(group.meteor.mean()),
        "context_attributable_fabrication_fraction": float(
            group.context_attributable_fabrication_fraction.mean()),
    }

rng = np.random.default_rng(int(config["seed"]) + 91)
bootstrap_rows = []
if BOOTSTRAPS > 0:
    for keys, group in detailed_frame.groupby(["model_key", "condition"], sort=True):
        values = {metric: [] for metric in grouped_metrics(group)}
        for _ in range(BOOTSTRAPS):
            sampled = group.iloc[rng.integers(0, len(group), size=len(group))]
            result = grouped_metrics(sampled)
            for metric, value in result.items():
                values[metric].append(value)
        for metric, samples in values.items():
            bootstrap_rows.append({
                "model_key": keys[0], "condition": keys[1], "metric": metric,
                "bootstrap_samples": BOOTSTRAPS,
                "ci_low": float(np.quantile(samples, 0.025)),
                "ci_high": float(np.quantile(samples, 0.975)),
            })
pd.DataFrame(bootstrap_rows).to_csv(
    REPORT_ROOT / "ecg_rag_metric_confidence_intervals.csv", index=False)


In [ ]:
comparison_pairs = []
for condition in config["conditions"]:
    if condition != "zero_shot":
        comparison_pairs.append((condition, "zero_shot", "rag_vs_zero_shot"))
for method in ["clip", "codebind", "combined"]:
    comparison_pairs.extend([
        (f"{method}_ecg_only_rag", "zero_shot", "ecg_neighbor_vs_zero_shot"),
        (f"{method}_ecg_plus_cxr_rag", f"{method}_ecg_only_rag", "cxr_added_to_ecg_rag"),
        (f"{method}_cxr_only_rag", "random_cxr_rag", "retrieved_vs_random_cxr"),
    ])
for scope in ["ecg_only_rag", "cxr_only_rag", "ecg_plus_cxr_rag"]:
    comparison_pairs.extend([
        (f"codebind_{scope}", f"clip_{scope}", "codebind_vs_clip"),
        (f"combined_{scope}", f"clip_{scope}", "combined_vs_clip"),
        (f"combined_{scope}", f"codebind_{scope}", "combined_vs_codebind"),
    ])
comparison_pairs = list(dict.fromkeys(comparison_pairs))
lower_is_better = {
    "fabrication_rate", "fabrication_rate_micro", "omission_rate", "omission_rate_micro",
    "context_attributable_fabrication_fraction"}
paired_rows = []
for model_key, outer in detailed_frame.groupby("model_key", sort=True):
    by_condition = {condition: frame.set_index("sample_id", drop=False)
                    for condition, frame in outer.groupby("condition")}
    for condition_a, condition_b, family in comparison_pairs:
        if condition_a not in by_condition or condition_b not in by_condition:
            continue
        common_ids = sorted(set(by_condition[condition_a].index) & set(by_condition[condition_b].index))
        if not common_ids:
            continue
        left = by_condition[condition_a].loc[common_ids].reset_index(drop=True)
        right = by_condition[condition_b].loc[common_ids].reset_index(drop=True)
        point_a, point_b = grouped_metrics(left), grouped_metrics(right)
        samples = {metric: [] for metric in point_a}
        if BOOTSTRAPS > 0:
            for _ in range(BOOTSTRAPS):
                positions = rng.integers(0, len(common_ids), size=len(common_ids))
                metric_a, metric_b = grouped_metrics(left.iloc[positions]), grouped_metrics(right.iloc[positions])
                for metric in samples:
                    improvement = (metric_b[metric] - metric_a[metric]
                                   if metric in lower_is_better
                                   else metric_a[metric] - metric_b[metric])
                    samples[metric].append(improvement)
        for metric in point_a:
            improvement = (point_b[metric] - point_a[metric]
                           if metric in lower_is_better else point_a[metric] - point_b[metric])
            values = samples[metric]
            paired_rows.append({
                "model_key": model_key, "comparison_family": family,
                "condition_a": condition_a, "condition_b": condition_b,
                "metric": metric, "n_paired": len(common_ids),
                "value_a": point_a[metric], "value_b": point_b[metric],
                "improvement_a_over_b": float(improvement),
                "ci_low": float(np.quantile(values, 0.025)) if values else np.nan,
                "ci_high": float(np.quantile(values, 0.975)) if values else np.nan,
                "bootstrap_probability_improvement": float(np.mean(np.asarray(values) > 0))
                if values else np.nan,
            })
paired_columns = [
    "model_key", "comparison_family", "condition_a", "condition_b", "metric",
    "n_paired", "value_a", "value_b", "improvement_a_over_b", "ci_low", "ci_high",
    "bootstrap_probability_improvement"]
paired = pd.DataFrame(paired_rows, columns=paired_columns)
paired.to_csv(REPORT_ROOT / "ecg_rag_paired_comparisons.csv", index=False)
primary_metrics = {"micro_f1", "fabrication_rate", "omission_rate"}
ecg_neighbor_vs_zero = paired.comparison_family.eq("ecg_neighbor_vs_zero_shot")
cxr_vs_zero = (
    paired.comparison_family.eq("rag_vs_zero_shot") &
    paired.condition_a.str.endswith("_cxr_only_rag"))
cxr_added = paired.comparison_family.eq("cxr_added_to_ecg_rag")
retrieved_vs_random = paired.comparison_family.eq("retrieved_vs_random_cxr")
codebind_cxr_vs_clip = (
    paired.comparison_family.eq("codebind_vs_clip") &
    paired.condition_a.eq("codebind_cxr_only_rag") &
    paired.condition_b.eq("clip_cxr_only_rag"))
primary_contrasts = paired[
    paired.metric.isin(primary_metrics) &
    (ecg_neighbor_vs_zero | cxr_vs_zero | cxr_added |
     retrieved_vs_random | codebind_cxr_vs_clip)]
primary_contrasts.to_csv(REPORT_ROOT / "ecg_rag_primary_contrasts.csv", index=False)
display(primary_contrasts)


In [ ]:
context_quality_rows = []
for row in contexts:
    sample_id = str(row["sample_id"])
    query_labels = reference_ecg_labels(sample_id)
    expected_cxr = set().union(*(ECG_TO_CXR_RELATIONS.get(label, set()) for label in query_labels))
    same_flags, cross_flags = [], []
    for item in row["retrieved"]:
        source_id = str(item["sample_id"])
        if item["modality"] == "ecg":
            source = extract_ecg_present(manifest_lookup["ecg"][source_id]["machine_diagnosis"])
            same_flags.append(bool(query_labels & source))
        elif expected_cxr:
            cross_flags.append(bool(expected_cxr & reference_cxr_labels(source_id)))
    context_quality_rows.append({
        "condition": row["condition"], "retrieval_method": row["retrieval_method"],
        "context_scope": row["context_scope"], "sample_id": sample_id,
        "same_label_hit_at_k": float(any(same_flags)) if same_flags else np.nan,
        "same_label_overlap_fraction": float(np.mean(same_flags)) if same_flags else np.nan,
        "cxr_relation_hit_at_k": float(any(cross_flags)) if cross_flags else np.nan,
        "cxr_relation_overlap_fraction": float(np.mean(cross_flags)) if cross_flags else np.nan,
        "cxr_relation_evaluable": bool(expected_cxr),
    })
context_quality = pd.DataFrame(context_quality_rows)
context_quality.to_csv(REPORT_ROOT / "ecg_rag_context_quality_detailed.csv", index=False)
context_quality_summary = context_quality.groupby(
    ["condition", "retrieval_method", "context_scope"], as_index=False).agg(
    n_queries=("sample_id", "size"),
    same_label_hit_at_k=("same_label_hit_at_k", "mean"),
    same_label_overlap_fraction=("same_label_overlap_fraction", "mean"),
    cxr_relation_hit_at_k=("cxr_relation_hit_at_k", "mean"),
    cxr_relation_overlap_fraction=("cxr_relation_overlap_fraction", "mean"),
    cxr_relation_evaluable=("cxr_relation_evaluable", "sum"))
context_quality_summary.to_csv(REPORT_ROOT / "ecg_rag_context_quality_summary.csv", index=False)
display(context_quality_summary)


In [ ]:
if sns is not None:
    sns.set_theme(style="whitegrid", context="talk")
else:
    plt.style.use("seaborn-v0_8-whitegrid")
order = config["conditions"]
model_order = list(config["models"])

def horizontal_metric_plot(metrics, title, filename, xlim=None):
    fig, axes = plt.subplots(1, len(model_order), figsize=(18, 10), sharey=True)
    axes = np.atleast_1d(axes)
    legend_handles, legend_labels = [], []
    for axis, model_key in zip(axes, model_order):
        model_frame = summary[summary.model_key.eq(model_key)].set_index("condition")
        model_frame = model_frame.reindex(order)
        if sns is not None:
            frame = model_frame.reset_index().melt(
                id_vars="condition", value_vars=metrics,
                var_name="metric", value_name="value")
            sns.barplot(
                data=frame, y="condition", x="value", hue="metric",
                order=order, ax=axis)
        else:
            positions = np.arange(len(order), dtype="float64")
            height = 0.8 / len(metrics)
            for metric_index, metric in enumerate(metrics):
                offset = (metric_index - (len(metrics) - 1) / 2) * height
                axis.barh(
                    positions + offset, model_frame[metric].to_numpy(dtype="float64"),
                    height=height, label=metric)
            axis.set_yticks(positions, order); axis.invert_yaxis()
        handles, labels = axis.get_legend_handles_labels()
        if handles and not legend_handles:
            legend_handles, legend_labels = handles, labels
        panel_legend = axis.get_legend()
        if panel_legend is not None:
            panel_legend.remove()
        axis.set_title(model_key); axis.set_ylabel("")
        if xlim is not None:
            axis.set_xlim(*xlim)
    fig.suptitle(title, y=0.995)
    if legend_handles:
        fig.legend(
            legend_handles, legend_labels,
            loc="upper center", bbox_to_anchor=(0.5, 0.955),
            ncol=len(legend_labels), frameon=False,
            fontsize=11, handlelength=1.6, columnspacing=1.2)
    fig.tight_layout(rect=(0, 0, 1, 0.90))
    fig.savefig(REPORT_ROOT / filename, dpi=180, bbox_inches="tight")
    plt.show()

horizontal_metric_plot(
    ["macro_f1", "micro_f1", "hamming_accuracy"],
    "ECG generation: structured clinical-content performance",
    "ecg_rag_multilabel_metrics.png", (0, 1))
horizontal_metric_plot(
    ["fabrication_rate", "omission_rate"],
    "ECG generation: fabrication and omission (lower is better)",
    "ecg_rag_fabrication_omission.png", (0, 1))
horizontal_metric_plot(
    ["cider", "bertscore_f1", "rougeL_f1"],
    "ECG generation: text similarity", "ecg_rag_text_similarity.png")
horizontal_metric_plot(
    ["bleu1", "bleu4", "meteor"],
    "ECG generation: IEEE-comparable lexical metrics",
    "ecg_rag_bleu_meteor.png", (0, 1))
print("Saved ECG-focused evaluation outputs to", REPORT_ROOT)


## Interpretation rules

- The ECG-to-ECG neighbor conditions are the required active RAG baseline. They are inspired by
  ECG-ReGen but are not an exact reproduction because this study uses bridge embeddings and external
  machine diagnoses.
- The primary CXR augmentation test is CXR-only RAG versus zero-shot.
- The active-control test is ECG+CXR RAG versus ECG-only RAG at the same case budget.
- Retrieved CXR context must outperform random CXR context to demonstrate retrieval specificity.
- Positive values in paired-comparison `improvement_a_over_b` always mean condition A is better;
  fabrication and omission are direction-reversed internally.
- Do not claim CodeBind superiority unless its paired ECG generation outcome exceeds CLIP; current
  bridge retrieval results favor the continuous baseline.
- A text-similarity increase with worse fabrication is not a clinical improvement.
- BLEU-1, BLEU-4, and METEOR facilitate comparison with prior ECG report-generation studies but do
  not measure clinical correctness; interpret them with the ECG-label and fabrication/omission outcomes.
- CXR-context-attributable fabrication is a safety diagnostic, not proof of causality.
- Review `ecg_rag_extractor_coverage.csv` and the parser self-tests before interpreting label metrics; low extraction coverage or contradictory assertions invalidate automatic fabrication/omission conclusions.
- The ECG reference is a machine interpretation, not an adjudicated cardiologist report.
- Conduct blinded cardiologist review before making clinical utility or safety claims.
